In [1]:
import matplotlib
matplotlib.use('Agg')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob
import re
from scipy.optimize import curve_fit
from matplotlib.gridspec import GridSpec
import os

## 1. Metropolis-Hastings Algorithm

We propose generating $N$ samples at a fixed temperature $T$; for this, at each iteration $i \in [0, N]$:

1. A random microstate is initialized by invoking the `microstate` class.
2. The energy and magnetization of the microstate are calculated using the `calc_E()` and `calc_M()` methods respectively.
3. A "flip" of a randomly chosen spin in the array is performed; this is achieved with the `flip(int index)` method, where the index is a random integer $\in [0, L*L - 1]$.
4. The new energy and magnetization of the microstate are calculated, but not using the previous methods: the update is performed efficiently by computing only the contribution of the nearest neighbors to the flipped spin.
5. The acceptance rule is applied:

```c++
deltaE <= 0 || probability_distribution(gen) < std::exp(-beta * deltaE)
```

This step is the heart of the method:
    * If the energy change upon flipping a spin is negative, a lower-energy state has been found and the transition is accepted.
    * Otherwise, a uniformly distributed random number $p$ in the interval $[0,1]$ is generated and compared against the relative transition probability defined as: $P[\delta E] = e^{-\beta \Delta E}$; if $p < P[\delta E]$ the transition is accepted, otherwise it is rejected.

6. Finally, the data of the current microstate is saved and written in real time to a `.txt` file.

It is worth noting that the above algorithm is implemented in two stages:

* Burn-in: the algorithm is first run without saving microstate information; this ensures the system reaches thermal equilibrium before sampling states. This allows for a more efficient exploration of the sample space.

* Sampling: once thermal equilibrium is reached, microstates are sampled; these are saved every 100 steps to avoid correlations.

In [3]:
def extract_L_and_temperature(filename):
    match = re.search(r'Ising(\d+)_T(\d+\.\d+)_Samples\.txt', filename)
    if match:
        L = int(match.group(1))
        temperature = float(match.group(2))
        return L, temperature
    return None, None

In [4]:
# Gaussian function for fitting
def gaussian(x, mean, amplitude, stddev):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))

In [5]:
def detect_L(data_dir="../data"):
    """Scan data_dir for Ising sample files and return the detected L value."""
    files = glob(os.path.join(data_dir, "Ising*_Samples.txt"))
    L_values = sorted({L_val for f in files for L_val, _ in [extract_L_and_temperature(f)] if L_val is not None})
    if not L_values:
        raise FileNotFoundError(f"No Ising sample files found in '{data_dir}'")
    if len(L_values) > 1:
        print(f"Multiple L values found: {L_values}. Using L = {L_values[0]}.")
    return L_values[0]

# Data extraction from files generated with Monte Carlo
L = detect_L()

L_temp_files = []
for file in glob(f"../data/Ising{L}_T*_Samples.txt"):
    file_L, temp = extract_L_and_temperature(file)
    if file_L == L and temp is not None:
        L_temp_files.append((temp, file))

if not L_temp_files:
    raise FileNotFoundError(f"No valid files found for L = {L}")

L_temp_files.sort(key=lambda x: x[0])
temperatures = [t[0] for t in L_temp_files]
files_sorted  = [t[1] for t in L_temp_files]

print(f"Found {len(temperatures)} files for L = {L} with temperatures: {temperatures}")

Found 20 files for L = 12 with temperatures: [0.01, 0.22, 0.43, 0.64, 0.85, 1.06, 1.27, 1.48, 1.69, 1.9, 2.11, 2.32, 2.53, 2.74, 2.95, 3.16, 3.37, 3.58, 3.79, 4.0]


In [6]:
# Main plot
n_temps = len(temperatures)
if n_temps == 0:
    print("No temperatures to plot.")
    exit(1)

n_cols = min(4, n_temps)
n_rows = (n_temps + n_cols - 1) // n_cols

fig = plt.figure(figsize=(5*n_cols, 4*n_rows))
gs = GridSpec(n_rows, n_cols, hspace=0.4, wspace=0.3)

# Process each file
for idx, (temp, file_path) in enumerate(zip(temperatures, files_sorted)):
    print(f"Processing file: {file_path}, T = {temp}")
    
    # Read data
    data = []
    with open(file_path, 'r') as f:
        for line_num, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    # Last two values are energy and magnetization
                    energy = float(parts[-2])
                    magnetization = float(parts[-1])
                    data.append((energy, magnetization))
                except ValueError as e:
                    print(f"Error on line {line_num}: {e}")
                    continue
    
    if not data:
        print(f"Warning: No valid data found in {file_path}")
        continue
    
    # Convert to numpy arrays
    energies = np.array([d[0] for d in data])
    magnetizations = np.array([d[1] for d in data])
    
    print(f"T = {temp}: {len(energies)} samples, E_range = [{energies.min()}, {energies.max()}], M_range = [{magnetizations.min()}, {magnetizations.max()}]")
    
    # Create subplot
    row = idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    
    # Energy histogram
    counts, bins, patches = ax.hist(energies, bins=30, alpha=0.7, 
                                   color='skyblue', density=True, 
                                   label=f'T = {temp:.2f}')
    
    # Gaussian fit
    try:
        bin_centers = (bins[:-1] + bins[1:]) / 2
        # Filter zero-count bins to avoid fitting issues
        mask = counts > 0
        if np.sum(mask) > 3:  # We need at least 3 points for the fit
            popt, _ = curve_fit(gaussian, bin_centers[mask], counts[mask], 
                               p0=[np.mean(energies), np.max(counts), np.std(energies)])
            
            # Plot fit
            x_fit = np.linspace(bins[0], bins[-1], 1000)
            ax.plot(x_fit, gaussian(x_fit, *popt), 'r-', 
                    label=f'μ={popt[0]:.2f}\nσ={popt[2]:.2f}')
        else:
            print(f"Not enough points to fit Gaussian for T={temp}")
    except Exception as e:
        print(f"Could not fit Gaussian for T={temp}: {e}")
    
    ax.set_title(f'T = {temp:.2f}, L = {L}')
    ax.set_xlabel('Energy')
    ax.set_ylabel('Probability density')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Energy Distribution for Different Temperatures (Ising Model {L}x{L})')
plt.tight_layout()
os.makedirs('../plots', exist_ok=True)
plt.savefig(f'../plots/energy_distributions_L{L}.png', dpi=300, bbox_inches='tight')
plt.close()

# Magnetization figure
fig2 = plt.figure(figsize=(5*n_cols, 4*n_rows))
gs2 = GridSpec(n_rows, n_cols, hspace=0.4, wspace=0.3)

for idx, (temp, file_path) in enumerate(zip(temperatures, files_sorted)):
    # Read data
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    magnetization = float(parts[-1])
                    data.append(magnetization)
                except ValueError:
                    continue
    
    if not data:
        continue
        
    magnetizations = np.array(data)
    
    # Create subplot
    row = idx // n_cols
    col = idx % n_cols
    ax = fig2.add_subplot(gs2[row, col])
    
    # Magnetization histogram
    ax.hist(magnetizations, bins=30, alpha=0.7, color='lightcoral', density=True)
    
    ax.set_title(f'T = {temp:.2f}, L = {L}')
    ax.set_xlabel('Magnetization')
    ax.set_ylabel('Probability density')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Magnetization Distribution for Different Temperatures (Ising Model {L}x{L})')
plt.tight_layout()
plt.savefig(f'../plots/magnetization_distributions_L{L}.png', dpi=300, bbox_inches='tight')
plt.close()

Processing file: ../data/Ising12_T0.010_Samples.txt, T = 0.01
T = 0.01: 10000 samples, E_range = [-288.0, -288.0], M_range = [-144.0, -144.0]
Not enough points to fit Gaussian for T=0.01
Processing file: ../data/Ising12_T0.220_Samples.txt, T = 0.22


T = 0.22: 10000 samples, E_range = [-288.0, -288.0], M_range = [-144.0, -144.0]


Not enough points to fit Gaussian for T=0.22
Processing file: ../data/Ising12_T0.430_Samples.txt, T = 0.43
T = 0.43: 10000 samples, E_range = [-288.0, -236.0], M_range = [-80.0, 144.0]
Not enough points to fit Gaussian for T=0.43
Processing file: ../data/Ising12_T0.640_Samples.txt, T = 0.64


T = 0.64: 10000 samples, E_range = [-288.0, -280.0], M_range = [142.0, 144.0]
Not enough points to fit Gaussian for T=0.64
Processing file: ../data/Ising12_T0.850_Samples.txt, T = 0.85


T = 0.85: 10000 samples, E_range = [-288.0, -276.0], M_range = [-144.0, -140.0]
Not enough points to fit Gaussian for T=0.85
Processing file: ../data/Ising12_T1.060_Samples.txt, T = 1.06
T = 1.06: 10000 samples, E_range = [-288.0, -264.0], M_range = [136.0, 144.0]


...

T = 3.58: 10000 samples, E_range = [-184.0, -28.0], M_range = [-102.0, 98.0]
Processing file: ../data/Ising12_T3.790_Samples.txt, T = 3.79


T = 3.79: 10000 samples, E_range = [-164.0, -16.0], M_range = [-86.0, 98.0]
Processing file: ../data/Ising12_T4.000_Samples.txt, T = 4.0


T = 4.0: 10000 samples, E_range = [-196.0, -8.0], M_range = [-84.0, 94.0]


/tmp/ipykernel_24793/1979125586.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


/tmp/ipykernel_24793/1979125586.py:118: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
